In [11]:
!pip install xgboost lightgbm optuna

In [12]:
import numpy as np
import pandas as pd
import matplotlib as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV 
from sklearn.model_selection import cross_val_score


In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import optuna

print("라이브러리 불러오기 완료")

라이브러리 불러오기 완료


In [15]:
plt.rcParams["font.family"] = "Malgun Gothic"

plt.rcParams["axes.unicode_minus"] = False

In [16]:
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns= data.feature_names)

In [17]:
df["target_original"] = data.target

In [18]:
df["target"] = 1 - df["target_original"]

In [19]:
df["target_name"] = df["target"].map({
    0:"benign",
    1:"malignant"
})

In [20]:
print("데이터의 크기 (행, 열)", df.shape)

print("\n원본 target 기준 개수:")
print(df["target_original"].value_counts().sort_index())

print("\n수업용 target 기준 개수:")
print(df["target"].value_counts().sort_index())

print("\n수업용 target 이름 기준 개수:")
print(df["target_name"].value_counts().sort_index())


데이터의 크기 (행, 열) (569, 33)

원본 target 기준 개수:
target_original
0    212
1    357
Name: count, dtype: int64

수업용 target 기준 개수:
target
0    357
1    212
Name: count, dtype: int64

수업용 target 이름 기준 개수:
target_name
benign       357
malignant    212
Name: count, dtype: int64


In [21]:
X = df.drop(columns = ["target_original","target","target_name"])

y = df["target"]

print("X 크기:", X.shape)
print("y 크기:", y.shape)

X 크기: (569, 30)
y 크기: (569,)


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)
print("train의 크기:", X_train.shape)
print("test의 크기:", X_test.shape)
print("\ntrain 의 target 비율:")
print(y_train.value_counts(normalize=True).round(3))
print("\ntest 의 target 비율:")
print(y_test.value_counts(normalize=True).round(3))

train의 크기: (455, 30)
test의 크기: (114, 30)

train 의 target 비율:
target
0    0.626
1    0.374
Name: proportion, dtype: float64

test 의 target 비율:
target
0    0.632
1    0.368
Name: proportion, dtype: float64


In [28]:
def evaluate_classification_model(model_name, y_true, y_pred, y_proba):
    """
    분류 모델의 성능을 같은 기준으로 평가하는 함수입니다.
    
    이 수업에서는  target 을 다음 기준으로 사용합니다.
    
    0 = benign, 양성
    1 = malignant, 악성
    
    따라서 precision, recall, f1-score 는 
    malignant, 즉 1번 클래스를 기준으로 계산합니다.
    
    model_name : 모델 이름 (비교표에서 구분용)
    y_true     : 실제 정답
    y_pred     : 모델이 예측한 0/1 값
    y_proba    : malignant(=1)일 확률 (predict_proba(X)[:, 1])
    """
    accuracy = accuracy_score(y_true, y_pred)

    precision_malignant = precision_score(y_true, y_pred, pos_label = 1)

    recall_malignant = recall_score(y_true, y_pred, pos_label = 1)

    f1_malignant = f1_score(y_true, y_pred, pos_label = 1)

    roc_auc = roc_auc_score(y_true, y_proba)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    result = {
        "model_name": model_name,
        "accuracy": accuracy,
        "precision_malignant": precision_malignant,
        "recall_malignant": recall_malignant,
        "f1_malignant": f1_malignant,
        "roc_auc": roc_auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp        
    }
    
    return result

print("평가 함수 준비 완료")

평가 함수 준비 완료


In [29]:
xgb_base_model = XGBClassifier(
    n_estimators = 100,
    max_depth = 3,
    learning_rate = 0.1,
    random_state = 42,
    eval_metric ="logloss"
)

In [31]:
xgb_base_model.fit(X_train, y_train)

xgb_base_pred = xgb_base_model.predict(X_test)


xgb_base_proba = xgb_base_model.predict_proba(X_test)[:, 1]

In [33]:
xgb_base_result = evaluate_classification_model(
    model_name = "XGBoost Base",
    y_true = y_test,
    y_pred = xgb_base_pred,
    y_proba = xgb_base_proba
)
xgb_base_result

{'model_name': 'XGBoost Base',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.9966931216931217,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}

In [ ]:
print("=== Classification Report (XGBoost Base) ===")
print(classification_report(
    y_test,
    xgb_base_pred,
    target_names = ["benign(0)", "malgnant(1)"]
))

print("=== Confusion_matrix(XGBoost Base) ===")
print(confusion_matrix(y_test, xgb_base_pred))
print("\n[[TN, FP],")
print(" [FN, TP]] 순서입니다.")


# 45page 까지 진행

=== Classification Report (XGBoost Base) ===
              precision    recall  f1-score   support

   benign(0)       0.95      1.00      0.97        72
 malgnant(1)       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114

=== Confusion_matrix(XGBoost Base) ===
[[72  0]
 [ 4 38]]

[[TN, FP],
 [FN, TP]] 순서입니다.


In [ ]:
xgb_for_random_search = XGBClassifier(
    random_state = 42,
    eval_metrics = "logloss"
)

In [ ]:
param_distributions = {
    "n_estimators" : [50, 100, 150, 200],
    "max_depth" : [2, 3, 4, 5],
    "learning_rate" : [0.01, 0.03, 0.05, 0.1],
    "subsample" : [0.7, 0.8, 0.9, 1.0],
    cols
}

NameError: name 'y_true' is not defined